# Feature Engineering

- In the previous notebooks, I completed dataset understanding, EDA, and data cleaning validation.

- In this notebook, I will create model-ready features from the PaySim transaction dataset. The raw dataset has no missing values and no major cleaning issue, so the main task now is feature engineering.

- Since the dataset contains more than 6 million rows, chunk processing will be used to create and save the processed dataset without loading the complete file into memory at once.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
raw_path = Path("../data/raw/paysim_transactions.csv")
processed_path = Path("../data/processed/processed_transactions.csv")

processed_path.parent.mkdir(parents=True, exist_ok=True)

raw_path, processed_path

(WindowsPath('../data/raw/paysim_transactions.csv'),
 WindowsPath('../data/processed/processed_transactions.csv'))

The original dataset contains transaction amount, account balances, transaction type, sender account ID, receiver account ID, and fraud labels.

For model training, account IDs should not be used directly because they are high-cardinality identifiers. Instead, useful patterns will be extracted from them.

The following features will be created:

- `transaction_id`
- account prefix features
- merchant destination flag
- balance difference features
- balance error features
- amount-to-balance ratio features
- encoded transaction type columns

In [3]:
# Feature Engineering Function
def create_features(df):
    df = df.copy()

    df["origin_prefix"] = df["nameOrig"].str[0]
    df["dest_prefix"] = df["nameDest"].str[0]

    df["is_origin_customer"] = (df["origin_prefix"] == "C").astype(int)
    df["is_dest_customer"] = (df["dest_prefix"] == "C").astype(int)
    df["is_dest_merchant"] = (df["dest_prefix"] == "M").astype(int)

    df["origin_balance_diff"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["destination_balance_diff"] = df["newbalanceDest"] - df["oldbalanceDest"]

    df["origin_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    df["destination_balance_error"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

    df["abs_origin_balance_error"] = df["origin_balance_error"].abs()
    df["abs_destination_balance_error"] = df["destination_balance_error"].abs()

    df["amount_to_oldbalanceOrg_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
    df["amount_to_oldbalanceDest_ratio"] = df["amount"] / (df["oldbalanceDest"] + 1)

    df["is_zero_oldbalanceOrg"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["is_zero_oldbalanceDest"] = (df["oldbalanceDest"] == 0).astype(int)

    type_encoded = pd.get_dummies(df["type"], prefix="type", dtype=int)
    df = pd.concat([df, type_encoded], axis=1)

    df = df.drop(columns=["nameOrig", "nameDest", "origin_prefix", "dest_prefix"])

    return df

## Why These Features Are Useful

- Balance difference features help compare the transaction amount with the actual balance movement.

- Balance error features are useful because fraudulent transactions may create abnormal differences between expected balance movement and actual balance movement.

- Account prefix features help identify whether the transaction is going to a customer or merchant account.

- Transaction type encoding is required because machine learning models cannot directly use text categories such as `TRANSFER`, `CASH_OUT`, or `PAYMENT`.

In [4]:
# Create Sample Features for Testing
sample_df = pd.read_csv(raw_path, nrows=1000)
sample_features = create_features(sample_df)

sample_features.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,is_origin_customer,...,abs_destination_balance_error,amount_to_oldbalanceOrg_ratio,amount_to_oldbalanceDest_ratio,is_zero_oldbalanceOrg,is_zero_oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,1,...,9839.64,0.057834,9839.640000,0,1,0,0,0,1,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,1,...,1864.28,0.087731,1864.280000,0,1,0,0,0,1,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,1,...,181.00,0.994505,181.000000,0,1,0,0,0,0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,1,...,21363.00,0.994505,0.008545,0,0,0,1,0,0,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,1,...,11668.14,0.280788,11668.140000,0,1,0,0,0,1,0


In [5]:
sample_features.columns

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud',
       'is_origin_customer', 'is_dest_customer', 'is_dest_merchant',
       'origin_balance_diff', 'destination_balance_diff',
       'origin_balance_error', 'destination_balance_error',
       'abs_origin_balance_error', 'abs_destination_balance_error',
       'amount_to_oldbalanceOrg_ratio', 'amount_to_oldbalanceDest_ratio',
       'is_zero_oldbalanceOrg', 'is_zero_oldbalanceDest', 'type_CASH_IN',
       'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER'],
      dtype='str')

In [6]:
# Save Processed Data in Chunks
chunk_size = 500000
start_id = 1
first_chunk = True

for chunk in pd.read_csv(raw_path, chunksize=chunk_size):
    chunk = create_features(chunk)

    chunk.insert(0, "transaction_id", range(start_id, start_id + len(chunk)))
    start_id += len(chunk)

    chunk.to_csv(
        processed_path,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk
    )

    first_chunk = False
    print(f"Processed rows till transaction_id: {start_id - 1}")

print("Feature engineering completed.")

Processed rows till transaction_id: 500000
Processed rows till transaction_id: 1000000
Processed rows till transaction_id: 1500000
Processed rows till transaction_id: 2000000
Processed rows till transaction_id: 2500000
Processed rows till transaction_id: 3000000
Processed rows till transaction_id: 3500000
Processed rows till transaction_id: 4000000
Processed rows till transaction_id: 4500000
Processed rows till transaction_id: 5000000
Processed rows till transaction_id: 5500000
Processed rows till transaction_id: 6000000
Processed rows till transaction_id: 6362620
Feature engineering completed.


In [7]:
# Verify the processed data
processed_sample = pd.read_csv(processed_path, nrows=5)
processed_sample.head()

,transaction_id,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,...,abs_destination_balance_error,amount_to_oldbalanceOrg_ratio,amount_to_oldbalanceDest_ratio,is_zero_oldbalanceOrg,is_zero_oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,...,9839.64,0.057834,9839.640000,0,1,0,0,0,1,0
1,2,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,...,1864.28,0.087731,1864.280000,0,1,0,0,0,1,0
2,3,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,...,181.00,0.994505,181.000000,0,1,0,0,0,0,1
3,4,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,...,21363.00,0.994505,0.008545,0,0,0,1,0,0,0
4,5,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,...,11668.14,0.280788,11668.140000,0,1,0,0,0,1,0


In [8]:
# Processed data summary
total_rows = 0

for chunk in pd.read_csv(processed_path, chunksize=500000):
    total_rows += len(chunk)

print(f"Total processed rows: {total_rows}")

Total processed rows: 6362620


In [9]:
# Target Variable Summary
target_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(processed_path, usecols=["isFraud"], chunksize=500000):
    target_counts = target_counts.add(chunk["isFraud"].value_counts(), fill_value=0)

target_counts.astype(int)

isFraud
0    6354407
1       8213
dtype: int64

## Feature Engineering Summary

This notebook prepares the PaySim transaction dataset for fraud detection model training.

The raw dataset was already validated in the previous notebook, so this stage focused on transforming the data into a model-ready format instead of performing basic cleaning or row deletion.

Since the dataset contains **6,362,620 rows**, the processed dataset was created using chunk processing. This allowed the full dataset to be transformed safely without loading the complete CSV into memory at once.

### Key Work Completed

- Created a unique `transaction_id` for every transaction.
- Extracted account-type signals from `nameOrig` and `nameDest`.
- Created customer and merchant indicator features.
- Added origin and destination balance movement features.
- Created balance error features to capture abnormal transaction behavior.
- Added amount-to-balance ratio features.
- Created zero-balance indicators.
- Encoded the `type` column into model-readable binary columns.
- Removed high-cardinality account ID columns that should not be used directly for model training.

### Final Processed Dataset

The final feature-engineered dataset contains:

- **6,362,620 rows**
- **28 columns**

The processed file was saved at:

`data/processed/processed_transactions.csv`

This dataset is now ready for the model training stage, where fraud detection models will be trained and evaluated using metrics such as recall, precision, F1-score, confusion matrix, and false negatives.